In [1]:
import os
import json
import gzip
import pandas as pd
import numpy as np

import scipy.sparse as spsparse
import scipy.stats as spstats


from statsmodels.stats.multitest import multipletests as holm
import statsmodels.api as sm

import matplotlib.pylab as plt
import matplotlib.gridspec as gridspec
import matplotlib.transforms as transforms

import seaborn as sns


# Load the data

In [8]:
def data_preprocess_bootstrap(country_auc_df):

    country_auc_df = country_auc_df[country_auc_df['N']>=2].reset_index(drop=True)
    
    country_auc_df['STD'] = country_auc_df.groupby(['CitingCountry', 'CitedYear', 'CitedCountry'])['AUC'].transform('std')
    country_auc_df['AUC'] = country_auc_df.groupby(['CitingCountry', 'CitedYear', 'CitedCountry'])['AUC'].transform('mean')
    
    country_auc_df.drop_duplicates(['CitingCountry', 'CitedYear', 'CitedCountry'], inplace=True)
    del country_auc_df['isample']
    
    country_auc_df['zscore'] = (country_auc_df['AUC'] - 0.5)/country_auc_df['STD']
    country_auc_df['pvalue'] = spstats.norm.sf(np.abs(country_auc_df['zscore'].values))
    
    country_auc_df['significant'] = country_auc_df['pvalue']<=0.01

    self_auc = country_auc_df[country_auc_df['CitingCountry'] == country_auc_df['CitedCountry']].sort_values(by=['CitedYear', 'CitedCountry']).reset_index(drop=True)
    self_auc.rename(columns={'CitedYear':'Year', 'CitedCountry':'Country'}, inplace=True)


    return self_auc

In [ ]:
name = 'noselfauthor_aucboot'

if name =='noselfauthor_aucboot':
   country_auc_bootstrap = pd.read_csv(os.path.join('../data/raw/CountryData', 'oa_countrycites_noselfauthor_aucboot.csv.gz'))
   filename_suffix='bootstrap_noselfauthor'

elif name =='noselfauthoraff_aucboot':
   country_auc_bootstrap = pd.read_csv(os.path.join('../data/raw/CountryData', 'oa_countrycites_noselfauthoraff_aucboot.csv.gz'))
   filename_suffix='bootstrap_noselfauthoraff'

else:
    print('------ \n No file request \n ------- ')
    country_auc_bootstrap=pd.DataFrame()

if not country_auc_bootstrap.empty:
    self_auc = data_preprocess_bootstrap(country_auc_bootstrap)

# Statistics

In [16]:
print('---------- basic information: start year, end year, number of countries')

self_auc['Year'].min(), self_auc['Year'].max(), self_auc['Country'].nunique()

---------- basic information: start year, end year, number of countries


(1848, 2024, 221)

In [17]:
self_auc[self_auc['Year']==2024]

,CitingCountry,Year,Country,AUC,Cov,N,zscore,pvalue,significant
10842,AE,2024,AE,0.523044,0.000187,121,1.686765,0.045824,False
10843,AF,2024,AF,0.498368,0.001284,15,-0.045560,0.481831,False
10844,AG,2024,AG,0.500000,NaN,1,NaN,NaN,False
10845,AL,2024,AL,0.602494,0.003763,13,1.670890,0.047372,False
10846,AM,2024,AM,0.492840,0.000008,12,-2.464273,0.006865,True
...,...,...,...,...,...,...,...,...,...
10987,XK,2024,XK,0.497925,NaN,1,NaN,NaN,False
10988,YE,2024,YE,0.489022,0.000005,15,-4.740404,0.000001,True
10989,ZA,2024,ZA,0.530997,0.000045,558,4.640876,0.000002,True
10990,ZM,2024,ZM,0.572781,0.001809,22,1.711071,0.043534,False


# other variables

In [ ]:
def compile_dataset(self_auc):

     
    countryprodnorm = pd.read_csv('../data/raw/CountryData/oa_countrynormtopproductivity.csv.gz')
    countryprodnorm.rename(columns={'NormTopFrac':'normalized_frac_top'}, inplace=True)
    
    countryprodnorm['logNumPub'] = np.log10(countryprodnorm['NumPub']+1)
    self_auc = self_auc.merge(countryprodnorm, how='left', on=['Country', 'Year'])
  
    national_authorship_rate = pd.read_csv( "../data/raw/CountryData/oa_national_authorship_rate.csv.gz")
    
    national_authorship_rate['FracInternationalAuthors']=1-national_authorship_rate['FractionNationalAuthors']
    self_auc = self_auc.merge(national_authorship_rate, how='left', on=['Country', 'Year'])
     
    wbgdp=pd.read_csv('../data/raw/CountryData/worldbank_indicators_08272025.csv')
    del wbgdp['country_name']
    wbgdp.rename(columns={'country_code':'Country', 'year':'Year',
                          'Research and development expenditure (% of GDP)':'RND_per',
                      'Patent applications, nonresidents':'PAT_res', 'Patent applications, residents': 'PAT_nres',
                      'GDP (constant 2015 US$)':'GDP','GDP per capita (constant 2015 US$)':'GDP_PCAP',
                       'GNI (constant 2015 US$)': 'GNI','GNI per capita (constant 2015 US$)':'GNI_PCAP',
                      'Researchers in R&D (per million people)':'NResearchers',
                      'Population, Total':'Pop'}, inplace=True)
    
  
    wbgdp['PAT_total'] = wbgdp['PAT_nres'] + wbgdp['PAT_res']
    for c in ['GDP', 'GNI', 'GDP_PCAP', 'GNI_PCAP', 'PAT_nres', 'PAT_res', 'PAT_total', ]:   #'GNI_PCAP', 'GDP_PCAP', 'RND_per'
        wbgdp[c] = np.log10(wbgdp[c].astype(float))
        
    wbgdp['logPop']= np.log10(wbgdp['Pop'])
    
    self_auc = self_auc.merge(wbgdp, how='left', on=['Country', 'Year'])

    return self_auc

In [19]:
dfself_vars=compile_dataset(self_auc)


In [ ]:
dfself_vars=compile_dataset(self_auc)

dfself_vars.sort_values(by=['Country','Year'],inplace=True)
dfself_vars['zscore'] = dfself_vars['zscore'].replace(-np.inf, 0)

dfself_vars['pub_capita']=dfself_vars['NumPub']/dfself_vars['Pop']

def income():
    
    income=pd.read_excel('../data/raw/CountryData/OGHIST_WorldBank.xlsx', sheet_name=2, skiprows=10)
    
    new_column_names = ['iso3','country']+list(range(1987, 2024))
    income.columns = new_column_names
    income['iso3']=income['iso3'].str.strip()
    del income['country']
    
    country_codes=pd.read_csv('../data/raw/CountryData/Gravity_csv_V202211/Countries_V202211.csv')
    country_codes['iso2']=country_codes['iso2'].str.strip()
    country_codes['iso3']=country_codes['iso3'].str.strip()
    country_dict = dict(zip(country_codes['iso3'], country_codes['iso2']))
    income['iso3'] = income['iso3'].map(country_dict)
    
    income=income[income['iso3'].notnull()]
    long_data = pd.melt(income, id_vars=["iso3"], var_name="Year", value_name="income_level")
    long_data.rename(columns={'iso3':'Country'}, inplace=True)

    return long_data

income=income()
    
dfself_vars=dfself_vars.merge(income, how='left', on=['Year','Country'])


dfself_vars.to_csv(f'../data/clean/SelfCitation_Plus_Covariates_{filename_suffix}_07172025.csv', index=False, sep=',')

In [22]:
dfself_vars.columns

Index(['CitingCountry', 'Year', 'Country', 'AUC', 'Cov', 'N', 'zscore',
       'pvalue', 'significant', 'NumPub', 'TopJournal', 'FracTop', 'OANumPub',
       'OATopJournal', 'OAFracTop', 'normalized_frac_top', 'logNumPub',
       'FractionNationalAuthors', 'FracInternationalAuthors',
       'TopicDiversity1', 'TopicDiversity2', 'RND_per', 'PAT_res', 'PAT_nres',
       'GDP', 'GDP_PCAP', 'GNI', 'GNI_PCAP', 'NResearchers', 'Pop',
       'PAT_total', 'logPop', 'pub_capita', 'income_level'],
      dtype='object')

In [ ]:
# add more control variables

In [ ]:
bootstrap=True

if bootstrap==False:
    
    dfself_vars=pd.read_csv('../data/clean/SelfCitation_Plus_Covariates_noselfauthor_07172025.csv')
    num_source_nations=dfself_vars['Country'].nunique()

if bootstrap==True:
    
    dfself_vars=pd.read_csv('../data/clean/SelfCitation_Plus_Covariates_bootstrap_noselfauthor_07172025.csv')
    num_source_nations=dfself_vars['Country'].nunique()


print(f' ----- number of citing nations: {num_source_nations}')

# Democracy

In [ ]:
def add_polity2(dfself_vars):
    
    polity=pd.read_excel('../data/raw/CountryData/p5v2018.xls')
    
    dff=pd.read_csv('../data/raw/CountryData/Jianjian/polity_country_revised.csv')
    s2iso=dict(zip(dff['scode'],dff['country_y']))
    
    dfp=polity[['scode','polity2','year']]
    dfp['scode']=dfp['scode'].map(s2iso)
    dfp.dropna(inplace=True)
    
    # because in 2011 Sudan has two values
    dfp=dfp.sort_values(by=['scode','year','polity2'],ascending=False).drop_duplicates(subset=['scode','year'])

    dfself_vars = dfself_vars.merge(dfp, how='left', right_on=['scode','year'],left_on=['Country','Year'])
    
    del dfself_vars['year']
    del dfself_vars['scode']

    dfself_vars['is_democratic'] = np.where(dfself_vars['polity2'].isna(), 
                                          np.nan,
                                          np.where(dfself_vars['polity2'] > 0, 1, 0))

    
    return dfself_vars

In [ ]:
dfself_vars = add_polity2(dfself_vars)

# Goverment index

In [ ]:
def add_goverment_index(dfself_vars):
    
    country_codes=pd.read_csv('../data/raw/CountryData/Gravity_csv_V202211/Countries_V202211.csv')
    country_codes['iso2']=country_codes['iso2'].str.strip()
    country_codes['iso3']=country_codes['iso3'].str.strip()
    country_dict = dict(zip(country_codes['iso3'], country_codes['iso2']))
    
    
    indexsum=pd.read_csv('../data/raw/CountryData/panel_sum_index_1990_2020.csv')
    indexsum.sort_values(by=['country_code','date'], inplace=True)
    indexsum=indexsum[['country_code','date','cap_sum_index', 'gov_sum_index',]]
    indexsum['country_code']=indexsum['country_code'].str.strip()
    indexsum['country_code'] = indexsum['country_code'].map(country_dict)
    
    data_index=dfself_vars.merge(indexsum, how='left',left_on=['Country','Year'], right_on=['country_code','date'])

    return data_index

In [ ]:
data_index= add_goverment_index(dfself_vars)

In [ ]:
bootstrap=True

if bootstrap==False:
    
    data_index.to_csv('../data/clean/SelfCitation_Plus_Covariates_noselfauthor_12272025.csv')

if bootstrap==True:
    
    data_index.to_csv('../data/clean/SelfCitation_Plus_Covariates_bootstrap_noselfauthor_12272025.csv')

# Data preprocess for regression in R

In [ ]:
def data_preprocess(name, numpub, start_year, end_year, sig):

    if name=='bootstrap_noselfauthoraff':
        dfself_vars=pd.read_csv('../data/clean/SelfCitation_Plus_Covariates_bootstrap_noselfauthoraff_07172025.csv')
    
    elif name=='bootstrap_noselfauthor':
        dfself_vars=pd.read_csv('../data/clean/SelfCitation_Plus_Covariates_bootstrap_noselfauthor_12272025.csv')
    else:
        dfself_vars=pd.DataFrame()
        return dfself_vars

        
    dfself_vars=dfself_vars[(dfself_vars['Year']<=end_year)&(dfself_vars['Year']>=start_year)]
    print(f'number of countries from 1990 to 2019: {dfself_vars['Country'].nunique()}')

    dfself_vars['Year']=dfself_vars['Year'].astype(int)
    
    dfself_vars = dfself_vars[dfself_vars['N']>=numpub]
    print(f'number of countries without nan and pub>50: {dfself_vars['Country'].nunique()}')

    
    conditions = [
        (dfself_vars['significant'] == True) & (dfself_vars['zscore'] < 0),
        (dfself_vars['significant'] == True) & (dfself_vars['zscore'] > 0),
        (dfself_vars['significant'] == False)
    ]
    
    choices = [-1, 1, 0]
    
    dfself_vars['sig_direction'] = np.select(conditions, choices, default=np.nan)

    dfself_vars['developed']=np.where(dfself_vars['income_level']=='H', 1,0)
    dfself_vars['income_level']=dfself_vars['income_level'].replace({'..':np.nan})
    dfself_vars['income_group'] = np.where(dfself_vars['income_level'].isin(['LM', 'L']), 'LM-L', dfself_vars['income_level'])

    if sig==True:
        dfself_vars=dfself_vars[dfself_vars['sig_direction']==1]

    if dfself_vars['zscore'].min()<=0:
        dfself_vars['logzscore']=np.log10(dfself_vars['zscore']-dfself_vars['zscore'].min()+1)
    else:
        dfself_vars['logzscore']=np.log10(dfself_vars['zscore'])


    return dfself_vars

filename_suffix='bootstrap_noselfauthor'
dfself_vars=data_preprocess(filename_suffix,50,1990,2019,False)
    
dfself_vars.to_csv(f'../data/clean/{filename_suffix}_R_12272025.csv', index=False, sep=',')

# Add disruption rate, novelty rate and hit rate

In [ ]:
df_final=pd.read_csv('../data/clean/country_novelty_disruption.csv.gz')
del df_final['num_pubs']

dfself=pd.read_csv("../data/clean/bootstrap_noselfauthor_R_12272025.csv")

del dfself['Unnamed: 0']
del dfself['country_code'] 

In [ ]:
dfmerge=dfself.merge(df_final, how='left', left_on=['Country','Year'], right_on=['country_code','year'])
del dfmerge['country_code']
del dfmerge['year']

dfhit=pd.read_csv('../data/clean/hit_rate_per_country_year.csv.gz')
dfhit_scinet=pd.read_csv('../data/clean/hit_rate_per_country_year_scinet.csv.gz')
dfhit_scinet.rename(columns={'hit_rate':'hit_rate_scinet'}, inplace=True)

dfmerge=dfmerge.merge(dfhit[['Country','Year','hit_rate']], how='left', on=['Country','Year'])

dfmerge=dfmerge.merge(dfhit_scinet[['country_code','year','hit_rate_scinet']], 
                      how='left', left_on=['Country','Year'], right_on=['country_code', 'year'])
del dfmerge['country_code']
del dfmerge['year']

dfmerge.to_csv('../data/clean/bootstrap_noselfauthor_R_disruption_03172026.csv', 
               index=False, sep=',')